In [1]:
from pathlib import Path
import json
from pyspark.sql import SparkSession

# SparkSession – Windows-safe configuration
spark = (
    SparkSession.builder
        .appName("nyc-taxi-etl")
        .config("spark.hadoop.io.nativeio.native.disable", "true")
        .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem")
        .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# Folders and manifest
PARQUET_DIR = Path("data/inbox/")
STATE_DIR = Path("state")
STATE_DIR.mkdir(parents=True, exist_ok=True)
STATE_FILE = STATE_DIR / "manifest.json"

if STATE_FILE.exists() and STATE_FILE.stat().st_size > 0:
    with open(STATE_FILE, "r") as f:
        processed_files = json.load(f).get("processed_files", [])
else:
    processed_files = []

files = list(PARQUET_DIR.glob("*.parquet"))
print("Inbox files:", [f.name for f in files])
print("Already processed:", processed_files)


Inbox files: ['yellow_tripdata_2025-01.parquet', 'yellow_tripdata_2025-02.parquet']
Already processed: []


In [2]:
from pyspark.sql.functions import col, max as spark_max, add_months, broadcast

def enrich(df, spark, lookup_path="data/lookup/taxi_zone_lookup.parquet", months=None):

    lookup = spark.read.parquet(lookup_path).cache()

    # Pickup enrichment
    pickup_lookup = lookup.select(
        col("LocationID").alias("pickup_LocationID"),
        col("Zone").alias("pickup_zone"),
        col("Borough").alias("pickup_borough"),
        col("service_zone").alias("pickup_service_zone")
    )

    df = df.join(
        broadcast(pickup_lookup),
        df.PULocationID == pickup_lookup.pickup_LocationID,
        "left"
    ).drop("pickup_LocationID")

    # Dropoff enrichment
    dropoff_lookup = lookup.select(
        col("LocationID").alias("dropoff_LocationID"),
        col("Zone").alias("dropoff_zone"),
        col("Borough").alias("dropoff_borough"),
        col("service_zone").alias("dropoff_service_zone")
    )

    df = df.join(
        broadcast(dropoff_lookup),
        df.DOLocationID == dropoff_lookup.dropoff_LocationID,
        "left"
    ).drop("dropoff_LocationID")

    # Scenario: last N months
    if months is not None and months > 0:
        cutoff = df.agg(
            add_months(spark_max("tpep_pickup_datetime"), -months).alias("cutoff")
        ).first()["cutoff"]

        if cutoff is not None:
            df = df.filter(col("tpep_pickup_datetime") >= cutoff)

    return df


In [3]:
from pyspark.sql.functions import (
    col, to_timestamp, unix_timestamp, to_date,
    input_file_name, current_timestamp
)

def transform(df):
    print("\n=== TRANSFORMATION START ===")
    print("Initial schema:")
    df.printSchema()
    print("\nInitial row count:", df.count())

    # 1. Parse timestamps
    print("\n-- Parsing timestamps --")
    df = df.withColumn("tpep_pickup_datetime", to_timestamp("tpep_pickup_datetime"))
    df = df.withColumn("tpep_dropoff_datetime", to_timestamp("tpep_dropoff_datetime"))
    print("After timestamp parsing:", df.count())

    # 2. Cleaning rules
    print("\n-- Cleaning rules --")

    bad_distance = df.filter(col("trip_distance") < 0)
    print("Bad rows (negative distance):", bad_distance.count())
    bad_distance.show(5, truncate=False)

    bad_passengers = df.filter(col("passenger_count") < 0)
    print("Bad rows (negative passenger_count):", bad_passengers.count())
    bad_passengers.show(5, truncate=False)

    df = df.filter(col("trip_distance") >= 0)
    df = df.filter(col("passenger_count") >= 0)
    df = df.filter(col("tpep_pickup_datetime").isNotNull())
    df = df.filter(col("tpep_dropoff_datetime").isNotNull())
    print("After cleaning:", df.count())

    # 3. Deduplication
    print("\n-- Deduplication --")
    before = df.count()
    df = df.dropDuplicates([
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID"
    ])
    after = df.count()
    print(f"Removed {before - after} duplicate rows")

    # 4. Derived fields
    print("\n-- Derived fields --")
    df = df.withColumn(
        "trip_duration_minutes",
        (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60
    )
    df = df.withColumn("pickup_date", to_date("tpep_pickup_datetime"))
    print("After derived fields:", df.count())

    # 5. Metadata
    print("\n-- Metadata columns --")
    df = df.withColumn("source_file", input_file_name())
    df = df.withColumn("ingested_at", current_timestamp())
    print("After metadata:", df.count())

    print("\nSample rows after transformation:")
    df.show(5, truncate=False)

    print("=== TRANSFORMATION END ===\n")
    return df


In [4]:
from pyspark.sql import DataFrame
import time

OUTPUT_PATH = Path("data/outbox/trips_enriched.parquet")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

N_MONTHS = None  # Example: set to 3 for last 3 months

etl_start = time.time()

for file_path in files:
    file_name = file_path.name

    if file_name in processed_files:
        print(f"{file_name}: already processed, skipping.")
        continue

    print(f"\n=== Processing: {file_name} ===")

    df = spark.read.parquet(str(file_path))

    df = transform(df)

    df = enrich(df, spark, months=N_MONTHS)

    if df.rdd.isEmpty():
        print(f"{file_name}: transformed DataFrame is empty, skipping write.")
        continue

    df.write.mode("append").parquet(str(OUTPUT_PATH))
    print(f"{file_name}: appended to {OUTPUT_PATH}")

    processed_files.append(file_name)
    with open(STATE_FILE, "w") as f:
        json.dump({"processed_files": processed_files}, f)

    print(f"{file_name}: manifest updated.")

etl_end = time.time()
print(f"\nTotal ETL runtime: {etl_end - etl_start:.2f} seconds")



=== Processing: yellow_tripdata_2025-01.parquet ===

=== TRANSFORMATION START ===
Initial schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: do

In [5]:
out_df = spark.read.parquet(str(OUTPUT_PATH))
print("Final output row count:", out_df.count())
out_df.show(5)


Final output row count: 5604161
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+-----------+-----------+--------------------+--------------------+--------------+-------------------+--------------+---------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|trip_duration_minutes|pickup_date|source_file|         ingested_at|         pickup_zone|pickup_borough|pickup_service_zone|  dropoff_zone|dropoff_borough|dropoff_service_zone|
+--------+--------------------+-------